**Create DataFrame**


In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [0]:
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000),
(103,"Rahul Sharma","Mumbai","Dermatology",1500),
(104,"Priya Nair","Bangalore","Cardiology",5000),
(105,"Vikram Singh","Chennai","Neurology",7000)
]

columns = ["visit_id","patient_name","city","department","consultation_fee"]

df = spark.createDataFrame(data, columns)

display(df)

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
102,Sneha Kapoor,Delhi,Orthopedics,3000
103,Rahul Sharma,Mumbai,Dermatology,1500
104,Priya Nair,Bangalore,Cardiology,5000
105,Vikram Singh,Chennai,Neurology,7000


**Write Data as Parquet**

In [0]:
df.write \
.mode("overwrite") \
.parquet("/tmp/patient_parquet")

**Read Parquet Data**

In [0]:
parquet_df = spark.read.parquet("/tmp/patient_parquet")
display(parquet_df)

visit_id,patient_name,city,department,consultation_fee
104,Priya Nair,Bangalore,Cardiology,5000
101,Arjun Reddy,Hyderabad,Cardiology,5000
102,Sneha Kapoor,Delhi,Orthopedics,3000
103,Rahul Sharma,Mumbai,Dermatology,1500
105,Vikram Singh,Chennai,Neurology,7000


**Schema Inspection**

In [0]:
parquet_df.printSchema()

root
 |-- visit_id: long (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- department: string (nullable = true)
 |-- consultation_fee: long (nullable = true)



**Column Projection (Read Specific Columns)**

In [0]:
spark.read.parquet("/tmp/patient_parquet") \
.select("patient_name","city") \
.show()

+------------+---------+
|patient_name|     city|
+------------+---------+
|  Priya Nair|Bangalore|
| Arjun Reddy|Hyderabad|
|Sneha Kapoor|    Delhi|
|Rahul Sharma|   Mumbai|
|Vikram Singh|  Chennai|
+------------+---------+



**Filtering Data**

In [0]:
spark.read.parquet("/tmp/patient_parquet") \
.filter("consultation_fee > 3000") \
.show()

+--------+------------+---------+----------+----------------+
|visit_id|patient_name|     city|department|consultation_fee|
+--------+------------+---------+----------+----------------+
|     104|  Priya Nair|Bangalore|Cardiology|            5000|
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|
|     105|Vikram Singh|  Chennai| Neurology|            7000|
+--------+------------+---------+----------+----------------+



**Partitioned Parquet Write**

In [0]:
df.write \
.mode("overwrite") \
.partitionBy("city") \
.parquet("/tmp/patient_parquet_partitioned")

**Read Partitioned Data**

In [0]:
spark.read.parquet("/tmp/patient_parquet_partitioned").show()

+--------+------------+-----------+----------------+---------+
|visit_id|patient_name| department|consultation_fee|     city|
+--------+------------+-----------+----------------+---------+
|     103|Rahul Sharma|Dermatology|            1500|   Mumbai|
|     101| Arjun Reddy| Cardiology|            5000|Hyderabad|
|     102|Sneha Kapoor|Orthopedics|            3000|    Delhi|
|     105|Vikram Singh|  Neurology|            7000|  Chennai|
|     104|  Priya Nair| Cardiology|            5000|Bangalore|
+--------+------------+-----------+----------------+---------+



**Partition Pruning (Performance Concept)**

In [0]:
spark.read.parquet("/tmp/patient_parquet_partitioned") \
.filter("city = 'Hyderabad'") \
.show()

+--------+------------+----------+----------------+---------+
|visit_id|patient_name|department|consultation_fee|     city|
+--------+------------+----------+----------------+---------+
|     101| Arjun Reddy|Cardiology|            5000|Hyderabad|
+--------+------------+----------+----------------+---------+



**Append Mode**

In [0]:
new_data = [
(106,"Ananya Das","Kolkata","Orthopedics",3000)
]
new_df = spark.createDataFrame(new_data, columns)
new_df.write \
.mode("append") \
.parquet("/tmp/patient_parquet")

**Overwrite Mode**

In [0]:
df.write \
.mode("overwrite") \
.parquet("/tmp/patient_parquet")

**Create SQL Table on Parquet(using view, because of Unity Catalog restriction)**

In [0]:
%sql
CREATE OR REPLACE VIEW patient_parquet_view AS 
SELECT * FROM parquet.`dbfs:/tmp/patient_parquet`;

**Query Parquet (Using VIEW instead of TABLE)**

In [0]:
%sql
SELECT * FROM patient_parquet_view;

visit_id,patient_name,city,department,consultation_fee
103,Rahul Sharma,Mumbai,Dermatology,1500
101,Arjun Reddy,Hyderabad,Cardiology,5000
105,Vikram Singh,Chennai,Neurology,7000
104,Priya Nair,Bangalore,Cardiology,5000
102,Sneha Kapoor,Delhi,Orthopedics,3000


**Convert Parquet → Delta (Path-based, not table-based)**

In [0]:
%sql
CONVERT TO DELTA parquet.`dbfs:/tmp/patient_parquet`;

**Compare Parquet vs Delta**

In [0]:
%sql
UPDATE delta.`/tmp/patient_parquet`
SET consultation_fee = 6000
WHERE visit_id = 101;

num_affected_rows
1


In [0]:
%sql
UPDATE delta.`dbfs:/tmp/patient_parquet`
SET consultation_fee = 6000
WHERE visit_id = 101;

num_affected_rows
1


**Real Use Case Pattern**

In [0]:
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000),
(103,"Rahul Sharma","Mumbai","Dermatology",1500),
(104,"Priya Nair","Bangalore","Cardiology",5000),
(105,"Vikram Singh","Chennai","Neurology",7000)
]

columns = ["visit_id","patient_name","city","department","consultation_fee"]

df = spark.createDataFrame(data, columns)

display(df)

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
102,Sneha Kapoor,Delhi,Orthopedics,3000
103,Rahul Sharma,Mumbai,Dermatology,1500
104,Priya Nair,Bangalore,Cardiology,5000
105,Vikram Singh,Chennai,Neurology,7000


In [0]:
df.write \
.mode("overwrite") \
.parquet("/tmp/landing_patient_parquet")

In [0]:
landing_df = spark.read.parquet("/tmp/landing_patient_parquet")
display(landing_df)

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
102,Sneha Kapoor,Delhi,Orthopedics,3000
104,Priya Nair,Bangalore,Cardiology,5000
103,Rahul Sharma,Mumbai,Dermatology,1500
105,Vikram Singh,Chennai,Neurology,7000


In [0]:
high_value_df = landing_df.filter("consultation_fee > 4000")
display(high_value_df)

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
104,Priya Nair,Bangalore,Cardiology,5000
105,Vikram Singh,Chennai,Neurology,7000


In [0]:
df.write \
.mode("overwrite") \
.partitionBy("city") \
.parquet("/tmp/partitioned_patient_parquet")

In [0]:
spark.read.parquet("/tmp/partitioned_patient_parquet") \
.filter("city = 'Chennai'") \
.show()

+--------+------------+----------+----------------+-------+
|visit_id|patient_name|department|consultation_fee|   city|
+--------+------------+----------+----------------+-------+
|     105|Vikram Singh| Neurology|            7000|Chennai|
+--------+------------+----------+----------------+-------+



In [0]:
new_data = [(106,"Ananya Das","Kolkata","Orthopedics",3000)]
new_df = spark.createDataFrame(new_data, columns)

new_df.write \
.mode("append") \
.parquet("/tmp/landing_patient_parquet")

In [0]:
%sql
CREATE OR REPLACE VIEW patient_parquet_view2 AS
SELECT * FROM parquet.`dbfs:/tmp/landing_patient_parquet`;

In [0]:
%sql
SELECT * FROM patient_parquet_view2;

In [0]:
%sql
CONVERT TO DELTA parquet.`dbfs:/tmp/landing_patient_parquet`;

In [0]:
%sql
UPDATE delta.`dbfs:/tmp/landing_patient_parquet`
SET consultation_fee = 6500
WHERE visit_id = 101;

num_affected_rows
1
